In [1]:
import pandas as pd
import numpy as np
import pickle

In [2]:
import os
print(os.listdir("crime_data/crime"))

['.ipynb_checkpoints', '01_District_wise_crimes_committed_IPC_2001_2012.csv', '01_District_wise_crimes_committed_IPC_2013.csv', '01_District_wise_crimes_committed_IPC_2014.csv', '02_01_District_wise_crimes_committed_against_SC_2001_2012.csv', '02_01_District_wise_crimes_committed_against_SC_2013.csv', '02_01_District_wise_crimes_committed_against_SC_2014.csv', '02_District_wise_crimes_committed_against_ST_2001_2012.csv', '02_District_wise_crimes_committed_against_ST_2013.csv', '02_District_wise_crimes_committed_against_ST_2014.csv', '03_District_wise_crimes_committed_against_children_2001_2012.csv', '03_District_wise_crimes_committed_against_children_2013.csv', '03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2012.csv', '03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2013.csv', '03_Persons_arrested_and_their_disposal_by_police_and_court_under_crime_against_children_2014.csv', '04_01_Person_arrested_and_the

In [3]:


# Load datasets
ipc = pd.read_csv("crime_data/crime/IPC_2001to2014.csv")
women1 = pd.read_csv("crime_data/crime/women_2001to 2013.csv")
women2 = pd.read_csv("crime_data/crime/42_District_wise_crimes_committed_against_women_2014.csv")
children = pd.read_csv("crime_data/crime/Children_2001to2013.csv")
police = pd.read_csv("crime_data/crime/12_Police_strength_actual_and_sanctioned.csv")

# Arrest data (multiple years)
arrests_2012 = pd.read_csv("crime_data/crime/04_01_Person_arrested_and_their_disposal_by_police_and_court_SLL_crime_2012.csv")
arrests_2013 = pd.read_csv("crime_data/crime/04_01_Person_arrested_and_their_disposal_by_police_and_court_SLL_crime_2013.csv")
arrests_2014 = pd.read_csv("crime_data/crime/04_01_Person_arrested_and_their_disposal_by_police_and_court_SLL_crime_2014.csv")

In [4]:
def clean(df):
    df.columns = df.columns.str.strip().str.lower()
    return df

def ensure_state_column(df):
    # Handle common variants after lowercasing column names
    aliases = ["state", "state/ut", "states/uts", "area_name"]
    for col in aliases:
        if col in df.columns:
            return df.rename(columns={col: "state"})
    raise KeyError(f"No state column found. Available columns: {df.columns.tolist()}")

def fix_state(df):
    df = ensure_state_column(df)
    df["state"] = df["state"].astype(str).str.strip().str.upper()
    df["state"] = df["state"].str.replace("&", "AND")
    return df

# Apply cleaning
ipc = fix_state(clean(ipc))
women1 = fix_state(clean(women1))
women2 = fix_state(clean(women2))
children = fix_state(clean(children))
police = fix_state(clean(police))

In [5]:
print(arrests_2012.columns)
print(arrests_2013.columns)
print(arrests_2014.columns)

Index(['STATE/UT', 'CRIME HEAD',
       'Persons in custody or on bail during the stage of investigation at the beginning of the year',
       'Persons arrested during the year',
       'Persons released or freed by Police or Magistrate before trial for want of evidence or any other reason',
       'Persons in custody or on bail during the stage of investigation at the end of the year',
       'Persons in whose cases charge sheets were laid during the year',
       'Persons under trial at the beginning of the year',
       'Total number of persons under trial during the year',
       'Persons against whom cases were compounded or withdrawn',
       'Persons in custody or on bail during the stage of trial at the end of the year',
       'Persons in whose cases trials were completed during the year',
       'Persons convicted', 'Persons acquitted'],
      dtype='object')
Index(['STATE/UT', 'CRIME HEAD',
       'Persons in custody or on bail during the stage of investigation at the beginn

In [6]:
print(arrests_2012.columns.tolist())
print(arrests_2013.columns.tolist())

['STATE/UT', 'CRIME HEAD', 'Persons in custody or on bail during the stage of investigation at the beginning of the year', 'Persons arrested during the year', 'Persons released or freed by Police or Magistrate before trial for want of evidence or any other reason', 'Persons in custody or on bail during the stage of investigation at the end of the year', 'Persons in whose cases charge sheets were laid during the year', 'Persons under trial at the beginning of the year', 'Total number of persons under trial during the year', 'Persons against whom cases were compounded or withdrawn', 'Persons in custody or on bail during the stage of trial at the end of the year', 'Persons in whose cases trials were completed during the year', 'Persons convicted', 'Persons acquitted']
['STATE/UT', 'CRIME HEAD', 'Persons in custody or on bail during the stage of investigation at the beginning of the year', 'Persons arrested during the year', 'Persons released or freed by Police or Magistrate before trial f

In [7]:
def extract_arrests(df, year):
    df.columns = df.columns.str.strip().str.lower()
    
    # CASE 1: already clean
    if "state" in df.columns and "total_arrests" in df.columns:
        df = df[["state", "total_arrests"]].copy()
    
    else:
        # rename state
        if "state/ut" in df.columns:
            df = df.rename(columns={"state/ut": "state"})
        
        # detect arrest column
        arrest_col = None
        
        if "persons arrested during the year" in df.columns:
            arrest_col = "persons arrested during the year"
        
        elif "persons arrested during the year_total" in df.columns:
            arrest_col = "persons arrested during the year_total"
        
        # fail safe
        if "state" not in df.columns or arrest_col is None:
            print(f"❌ ERROR in year {year}: ", df.columns.tolist())
            return pd.DataFrame(columns=["state","total_arrests","year"])
        
        df = df[["state", arrest_col]].copy()
        df.columns = ["state", "total_arrests"]
    
    # clean values
    df["state"] = df["state"].astype(str).str.strip().str.upper()
    df["total_arrests"] = pd.to_numeric(df["total_arrests"], errors="coerce").fillna(0)
    
    df["year"] = year
    
    return df

In [8]:
def standardize_state_column(df):
    df.columns = df.columns.str.strip().str.lower()
    
    if "state/ut" in df.columns:
        df = df.rename(columns={"state/ut": "state"})
    elif "states/uts" in df.columns:
        df = df.rename(columns={"states/uts": "state"})
    elif "area_name" in df.columns:
        df = df.rename(columns={"area_name": "state"})
    
    return df

In [9]:
arrests_2012 = standardize_state_column(arrests_2012)
arrests_2013 = standardize_state_column(arrests_2013)
arrests_2014 = standardize_state_column(arrests_2014)

a2012 = extract_arrests(arrests_2012, 2012)
a2013 = extract_arrests(arrests_2013, 2013)
a2014 = extract_arrests(arrests_2014, 2014)

In [10]:
arrests = pd.concat([a2012, a2013, a2014], ignore_index=True)

arrests = arrests.groupby(["state","year"])["total_arrests"].sum().reset_index()

In [11]:
ipc["ipc_total"] = ipc.select_dtypes(include="number").sum(axis=1)
ipc = ipc[["state","district","year","ipc_total"]]

women1["women_total"] = women1.select_dtypes(include="number").sum(axis=1)
women1 = women1[["state","district","year","women_total"]]

women = pd.concat([women1, women2], ignore_index=True)

children["children_total"] = children.select_dtypes(include="number").sum(axis=1)
children = children[["state","district","year","children_total"]]

police = police.rename(columns={"rank_all_ranks_total": "total_police"})
police = police.groupby(["state","year"])["total_police"].sum().reset_index()

In [12]:
df = ipc.merge(women, on=["state","district","year"], how="left")
df = df.merge(children, on=["state","district","year"], how="left")
df = df.merge(police, on=["state","year"], how="left")
df = df.merge(arrests, on=["state","year"], how="left")

df = df.fillna(0)
df = df[df["year"] >= 2012]

In [13]:
df["total_crime"] = df["ipc_total"] + df["women_total"] + df["children_total"]

df["crime_per_police"] = df["total_crime"] / (df["total_police"] + 1)
df["arrest_rate"] = df["total_arrests"] / (df["total_crime"] + 1)

df["crime_growth"] = df.groupby(["state","district"])["total_crime"].pct_change().fillna(0)

df["women_ratio"] = df["women_total"] / (df["total_crime"] + 1)
df["children_ratio"] = df["children_total"] / (df["total_crime"] + 1)

In [14]:
df["future_crime"] = df.groupby(["state","district"])["total_crime"].shift(-1)

df = df.dropna()

df["risk"] = pd.qcut(df["future_crime"], 3, labels=["Low","Medium","High"])

In [15]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

X = df[[
    "total_crime",
    "crime_per_police",
    "arrest_rate",
    "crime_growth",
    "women_ratio",
    "children_ratio",
    "year"
]]

y = df["risk"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=300, max_depth=12, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(max_depth=12, n_estimators=300, random_state=42)

In [16]:


pickle.dump(model, open("model.pkl", "wb"))
print("Model saved ✅")

Model saved ✅


In [17]:
from sklearn.metrics import classification_report

print("Accuracy:", model.score(X_test, y_test))
print(classification_report(y_test, model.predict(X_test)))

Accuracy: 0.961038961038961
              precision    recall  f1-score   support

        High       0.98      0.95      0.96        42
         Low       0.96      0.98      0.97        53
      Medium       0.95      0.95      0.95        59

    accuracy                           0.96       154
   macro avg       0.96      0.96      0.96       154
weighted avg       0.96      0.96      0.96       154



In [18]:
for feature, importance in zip(X.columns, model.feature_importances_):
    print(feature, round(importance, 3))

total_crime 0.311
crime_per_police 0.276
arrest_rate 0.041
crime_growth 0.002
women_ratio 0.159
children_ratio 0.209
year 0.001


In [19]:
def predict_by_state_overall(state):
    
    state = state.upper().strip()
    
    state_data = df[df["state"] == state]
    
    if state_data.empty:
        return "State not found", None
    
    # Aggregation
    total_crime = state_data["total_crime"].mean()
    crime_per_police = state_data["crime_per_police"].mean()
    arrest_rate = state_data["arrest_rate"].mean()
    crime_growth = state_data["crime_growth"].mean()
    women_ratio = state_data["women_ratio"].mean()
    children_ratio = state_data["children_ratio"].mean()
    
    # 🔥 ADD THIS (IMPORTANT)
    ipc_avg = state_data["ipc_total"].mean()
    women_avg = state_data["women_total"].mean()
    children_avg = state_data["children_total"].mean()
    
    year = state_data["year"].max()
    
    input_df = pd.DataFrame([[
        total_crime,
        crime_per_police,
        arrest_rate,
        crime_growth,
        women_ratio,
        children_ratio,
        year
    ]], columns=X.columns)
    
    risk = model.predict(input_df)[0]
    
    return risk, {
        "total_crime": total_crime,
        "crime_growth": crime_growth,
        "arrest_rate": arrest_rate,
        "ipc": ipc_avg,
        "women": women_avg,
        "children": children_avg
    }

In [20]:
def give_reason_overall(data, risk):
    
    # Find dominant crime type
    crime_types = {
        "IPC Crimes": data["ipc"],
        "Crimes Against Women": data["women"],
        "Crimes Against Children": data["children"]
    }
    
    dominant_crime = max(crime_types, key=crime_types.get)
    
    # Build reasoning
    if risk == "High":
        
        if data["crime_growth"] > 0:
            trend = "increasing trend"
        else:
            trend = "consistently high levels"
        
        if data["arrest_rate"] < 0.3:
            enforcement = "low arrest efficiency"
        else:
            enforcement = "moderate enforcement effectiveness"
        
        return (
            f"High crime risk due to {trend} in overall crime. "
            f"The state shows {enforcement}, with {dominant_crime} being the dominant contributor."
        )
    
    elif risk == "Medium":
        
        return (
            f"Moderate risk due to stable but noticeable crime levels. "
            f"{dominant_crime} contributes significantly to total crime, with moderate enforcement efficiency."
        )
    
    else:
        
        return (
            f"Low risk due to controlled and stable crime trends. "
            f"Although {dominant_crime} is the major category, overall crime remains under control."
        )

In [21]:
def give_suggestions(data, risk):
    
    suggestions = []
    
    # 🔥 Based on risk level
    if risk == "High":
        suggestions.append("Increase police deployment in high-crime districts")
        suggestions.append("Strengthen surveillance using CCTV and smart monitoring")
    
    elif risk == "Medium":
        suggestions.append("Increase patrolling in sensitive areas")
        suggestions.append("Monitor crime hotspots regularly")
    
    else:
        suggestions.append("Maintain current policing strategies")
        suggestions.append("Continue preventive monitoring")
    
    
    # 🔥 Based on dominant crime type
    crime_types = {
        "IPC Crimes": data["ipc"],
        "Women Crimes": data["women"],
        "Children Crimes": data["children"]
    }
    
    dominant = max(crime_types, key=crime_types.get)
    
    if dominant == "Women Crimes":
        suggestions.append("Enhance women safety programs and helplines")
        suggestions.append("Increase security in public places and transport")
    
    elif dominant == "Children Crimes":
        suggestions.append("Strengthen child protection units and awareness programs")
        suggestions.append("Improve school and community safety measures")
    
    else:
        suggestions.append("Focus on general crime prevention strategies and law enforcement")
    
    
    # 🔥 Based on arrest efficiency
    if data["arrest_rate"] < 0.3:
        suggestions.append("Improve investigation efficiency and arrest rates")
    
    
    # 🔥 Based on trend
    if data["crime_growth"] > 0:
        suggestions.append("Take immediate action to control rising crime trends")
    
    
    return suggestions

In [22]:
print("\n📍 Available States:")
for s in sorted(df["state"].unique()):
    print("-", s)

state = input("\nEnter State: ")

risk, data = predict_by_state_overall(state)

if risk == "State not found":
    print("❌ State not found")

else:
    print("\n🔍 OVERALL STATE ANALYSIS (MULTI-YEAR)")
    print("--------------------------------------")
    
    print("State:", state.upper())
    
    print("\n📊 Average Crime Count:", int(data["total_crime"]))
    
    print("\n⚠️ Overall Risk Level:", risk)
    
    # Reason
    reason = give_reason_overall(data, risk)
    print("\n🧠 Reason:")
    print(reason)
    
    # Crime Breakdown
    print("\n📊 Crime Breakdown:")
    print("IPC Crimes:", int(data["ipc"]))
    print("Women-related Crimes:", int(data["women"]))
    print("Children-related Crimes:", int(data["children"]))
    
    # Suggestions
    print("\n🚔 Recommendations:")
    suggestions = give_suggestions(data, risk)
    
    for s in suggestions:
        print("-", s)


📍 Available States:
- ANDHRA PRADESH
- ARUNACHAL PRADESH
- ASSAM
- BIHAR
- CHANDIGARH
- CHHATTISGARH
- DAMAN AND DIU
- DELHI UT
- GOA
- GUJARAT
- HARYANA
- HIMACHAL PRADESH
- JAMMU AND KASHMIR
- JHARKHAND
- KARNATAKA
- KERALA
- LAKSHADWEEP
- MADHYA PRADESH
- MAHARASHTRA
- MANIPUR
- MEGHALAYA
- MIZORAM
- NAGALAND
- ODISHA
- PUDUCHERRY
- PUNJAB
- RAJASTHAN
- SIKKIM
- TAMIL NADU
- TRIPURA
- UTTAR PRADESH
- UTTARAKHAND
- WEST BENGAL



Enter State:  BIHAR



🔍 OVERALL STATE ANALYSIS (MULTI-YEAR)
--------------------------------------
State: BIHAR

📊 Average Crime Count: 13587

⚠️ Overall Risk Level: Medium

🧠 Reason:
Moderate risk due to stable but noticeable crime levels. IPC Crimes contributes significantly to total crime, with moderate enforcement efficiency.

📊 Crime Breakdown:
IPC Crimes: 9208
Women-related Crimes: 2235
Children-related Crimes: 2143

🚔 Recommendations:
- Increase patrolling in sensitive areas
- Monitor crime hotspots regularly
- Focus on general crime prevention strategies and law enforcement


In [23]:
df.to_csv("final_data.csv", index=False)